---
**Data Cleanup & Visualization in Python**
Data Analysis Course · Week 2
---

This notebook is the Python equivalent of the R Markdown `_01_data_cleanup.Rmd`.
It uses the same diabetes dataset to practice **data cleanup** (removing columns/rows, reordering)
and **visualizing distributions** (histograms, density plots, boxplots, QQ-plots).

Work through it cell by cell — run each code cell with **Shift+Enter**.

**Required packages:** `pandas`, `numpy`, `matplotlib`, `scipy`
```
pip install pandas numpy matplotlib scipy
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## 1 – Loading and understanding the data *(equivalent to R section 2)*

In [ ]:
# R: dat <- read.delim(url)
dat = pd.read_csv(
    "https://www.dropbox.com/scl/fi/zqjdoi7naolmruyxrraxq/diabetes_2025.tsv?rlkey=txnps3ccr0vj47yvjmpecbeom&dl=1",
    sep="\t"
)
dat.head(10)   # R: head(dat, 10)

In [ ]:
# R: dim(dat)
print(dat.shape)              # (rows, columns)

# R: ncol(dat) / nrow(dat)
print(dat.shape[1])           # number of columns
print(dat.shape[0])           # number of rows

In [ ]:
# R: colnames(dat)
print(dat.columns.tolist())   # R equivalent for rows: dat.index

For more description of these values, [look here](https://hbiostat.org/data/repo/cdiabetes).

In [ ]:
# R: min(dat$age); max(dat$age); range(dat$age)
print(dat["age"].min())
print(dat["age"].max())
print(dat["age"].min(), dat["age"].max())   # "range"

# Can you find out the same for height and weight?

In [ ]:
# R: summary(dat)
dat.describe(include="all")

Can you explain what you see?

Before we keep going, here is a quick reminder of how to access columns, rows or individual cells
in a DataFrame. **If you're already familiar with it you can skip the next cell.**

In [ ]:
# Returning a specific column or row of a DataFrame
dat.iloc[0]        # Returns the first row     (R: dat[1,])
dat.iloc[:, 0]      # Returns the first column  (R: dat[,1])
dat.iloc[0, 0]      # Returns only the first cell of the first row (R: dat[1,1])

# If the columns have names, you can access a column using
dat["gender"]       # (R: dat$gender)

# Returning a range of columns or rows
dat.iloc[0:3, :]     # Returns the first three rows    (R: dat[1:3,])
dat.iloc[:, 0:3]     # Returns the first three columns (R: dat[,1:3])
dat.iloc[0:3, 0:3]   # Returns the first 3x3 block      (R: dat[1:3,1:3])

# Returning a value or a range of values in a Series
dat["age"].iloc[0]     # Returns the first value        (R: dat$age[1])
dat["age"].iloc[0:3]   # Returns the first three values (R: dat$age[1:3])
# Note: Python indexing is 0-based, R indexing is 1-based!

## 2 – Data cleanup *(equivalent to R section 3)*

Very often the first thing one needs to do before any data science project is to clean up the raw
data and transform it into a format that is readily understood and easy to use for downstream
analysis. This usually involves:

- Removing empty value rows/columns
- Removing unused or unnecessary rows/columns
- Reordering the data matrix
- Keeping columns uniformly numeric, string, or boolean
- Handling data-specific quirks (e.g. inconsistent separators in numbers)

Let's clean up our diabetes data:

1. Remove the `bp.2s` and `bp.2d` columns — mostly missing values (see `describe()` above)
2. Remove the column `time.ppn` — not required for our analysis
3. Reorder the columns so qualitative and quantitative values are separated, with related
   quantitative variables kept together

**IMPORTANT:** before any cleaning, we save the original data into a new variable, so we can go
back to it if needed.

In [ ]:
dat_original = dat.copy()   # R: dat.original <- dat

In [ ]:
# Which columns correspond to bp.2s, bp.2d and time.ppn?
cols_to_remove = ["bp.2s", "bp.2d", "time.ppn"]
print(cols_to_remove)
# R uses which(colnames(dat) %in% ...) to get column INDICES;
# in pandas we can just refer to columns by NAME directly with drop().

In [ ]:
# R: dat <- dat[, -i.remove]
dat = dat.drop(columns=cols_to_remove)

In [ ]:
# Reorder columns: categorical first, then numerical (grouped by topic)
column_order = [
    "gender", "location", "frame", "id", "chol",
    "stab.glu", "hdl", "ratio", "glyhb", "age",
    "height", "weight", "bp.1s", "bp.1d"
]
dat = dat[column_order]
# R used numeric positions (dat[,c(8,6,11,...)]); in Python it's clearer (and safer)
# to reorder by column NAME.

In [ ]:
dat.describe(include="all")   # R: summary(dat)

The ordering and selection of columns looks right, but some columns still have missing values
(e.g. `glyhb` has several `NaN`s). Let's remove all rows with any missing value.

Remember: 1 row = 1 patient.

In [ ]:
# R: is.na(dat[8,"bp.1s"])
print(dat.iloc[7]["bp.1s"])          # row 8 in R = index 7 in Python (0-based!)
print(pd.isna(dat.iloc[7]["bp.1s"]))

In [ ]:
# R: sum(is.na(dat[1,]))  — count NAs in the first row
print(dat.iloc[0].isna().sum())    # 0 — no missing values on this row
dat.iloc[0]

In [ ]:
# R: apply(dat, 1, function(x) sum(is.na(x)))
# axis=1 -> apply row-wise (same logic as R's MARGIN=1)
nb_na_rows = dat.isna().sum(axis=1)
nb_na_rows.head()

What should be the expected length of this Series? Think about the shape of `dat`.

In [ ]:
# R: i.missing = which(nb_NA.rows > 0)
i_missing = nb_na_rows[nb_na_rows > 0].index
i_missing[:6]

In [ ]:
# R: dat = dat[-i.missing,]
dat = dat.drop(index=i_missing)
# How many patients were removed because of missing values?
print(dat_original.shape[0] - dat.shape[0])

In [ ]:
dat.describe(include="all")   # cleaned data: no missing values, columns cleanly ordered

Can you identify which type of data (continuous, discrete, categorical, ...) each column above represents, and why?

## 3 – Visualizing data distribution *(equivalent to R section 4)*

### Histograms

In [ ]:
# R: hist(dat$stab.glu, ...)
plt.hist(dat["stab.glu"])
plt.xlabel("Stabilized Glucose concentration in blood")
plt.title("Glucose concentration")
plt.show()

# Try adding bins=50 and see what happens. Try 10, 20, 75, 100 and compare.

### Density plots

In [ ]:
# R: d <- density(dat$stab.glu, bw = 20); plot(d, ...)
dat["stab.glu"].plot(kind="density", bw_method=0.5)
plt.xlabel("Stabilized Glucose concentration in blood")
plt.title("Glucose concentration")
plt.show()

# Change bw_method and see how the plot changes.

### Boxplots

In [ ]:
# R: boxplot(dat$stab.glu, ..., horizontal = TRUE)
plt.boxplot(dat["stab.glu"].dropna(), vert=False)
plt.xlabel("Stabilized Glucose concentration in blood")
plt.show()

# Can you explain all features of this graph — upper/lower whisker, 25% quantile, ...?

### QQ-plots

In [ ]:
## Let's first make a histogram
plt.hist(dat["bp.1s"])
plt.show()

## Maybe with more bins?
plt.hist(dat["bp.1s"], bins=20)
plt.show()

In [ ]:
# R: qqnorm(dat$bp.1s); qqline(dat$bp.1s)
stats.probplot(dat["bp.1s"].dropna(), dist="norm", plot=plt)
plt.show()

# So — is the distribution normal?

Now let's compare the quantiles of the blood pressure values of men and women!

In [ ]:
# R used which() to select rows for men and women; in pandas we use boolean masks directly
bp_men = dat.loc[dat["gender"] == "male", "bp.1s"]
bp_women = dat.loc[dat["gender"] == "female", "bp.1s"]

# Compute the quantiles (dropna() first, same purpose as R's na.rm=TRUE)
probs = np.arange(0, 1.05, 0.05)
q_men = bp_men.dropna().quantile(probs)
q_women = bp_women.dropna().quantile(probs)

# Now plot against each other!
plt.scatter(q_men, q_women)
plt.xlabel("Men — bp.1s quantiles")
plt.ylabel("Women — bp.1s quantiles")
plt.show()

---
## Exercises

### Exercise 1: Data Exploration and Summary Statistics

`describe()` gives us a quick overview, but sometimes we need more specific information.

1. Calculate the mean and standard deviation of the `hdl` (High Density Lipoprotein) values for the
   entire dataset using `.mean()` and `.std()`.
2. Create a new column called `bmi` (Body Mass Index) using the formula:
   BMI = weight (in pounds) / height² (in inches) × 703. Add it with `dat["bmi"] = ...`
3. Use `.value_counts()` on the `location` column to count how many patients come from each
   location. Which location has the most patients?
4. Calculate the median age separately for male and female patients using `.groupby()`. Are they similar?

In [ ]:
# Your code here:

### Exercise 2: Visualizing Relationships Between Variables

1. Create a scatter plot of `weight` vs `height` using `plt.scatter()`. Do you see a relationship?
2. Color the points by gender (*hint: pass `c=dat["gender"].astype("category").cat.codes`*). Does
   the relationship appear different for men and women?
3. Use `.corr()` to calculate the correlation coefficient between weight and height. How strong is
   this relationship?
4. Create side-by-side boxplots comparing cholesterol levels (`chol`) between males and females
   using `dat.boxplot(column="chol", by="gender")`. What differences do you observe? Try other variables!

In [ ]:
# Your code here:

### Going further: Advanced Data Manipulation (for advanced students)

1. Create a categorical column called `age_group` dividing patients into three categories: "young"
   (age < 40), "middle" (40 ≤ age < 60), and "senior" (age ≥ 60).
   *Hint: use `pd.cut()`.*
2. Calculate the mean cholesterol level for each combination of `gender` and `age_group` using
   `.groupby([...]).mean()`.
3. Write a function `standardize(x)` that z-transforms a numeric Series (subtract mean, divide by
   standard deviation). Apply it to all numeric columns to create `dat_scaled`.
   *Hint: `dat.select_dtypes(include=np.number).apply(standardize)`.*
4. Using `dat_scaled`, identify which patient has the most extreme (highest or lowest) standardized
   BMI value. Print out all their clinical information. What makes this patient unusual?

In [ ]:
# Your code here:

## Summary: What have we learned?

| R | Python | Purpose |
|---|--------|---------|
| `dim(dat)` | `dat.shape` | Rows/columns |
| `colnames(dat)` | `dat.columns` | Column names |
| `summary(dat)` | `dat.describe(include="all")` | Summary statistics |
| `dat[, -i.remove]` | `dat.drop(columns=[...])` | Remove columns |
| `dat[-i.missing,]` | `dat.drop(index=[...])` | Remove rows |
| `is.na(x)` | `pd.isna(x)` / `x.isna()` | Detect missing values |
| `apply(dat, 1, f)` | `dat.apply(f, axis=1)` | Row-wise function |
| `which(x > 5)` | `x[x > 5].index` | Row selection by condition |
| `hist()` | `plt.hist()` | Histogram |
| `density()` + `plot()` | `Series.plot(kind="density")` | Density plot |
| `boxplot()` | `plt.boxplot()` / `df.boxplot()` | Boxplot |
| `qqnorm()` + `qqline()` | `scipy.stats.probplot()` | QQ-plot |
| `tapply()` | `.groupby()` | Group-wise aggregation |
| `cut()` | `pd.cut()` | Bin numeric values into categories |